# w9_twostage_5fold.ipynb: BYOL warm start, then the windowed teacher, five folds @pool (Table A12)

The two-stage row of the anchor-supply economies. Stage one is the BYOL tower of the same fold on
`pool` packs. Stage two initializes the tower from those weights (`--init-ckpt`, run names end in
`_bw`) and trains the windowed teacher `wcle_swin168step84loop2i2ce_icetf` for 600 epochs, 30% of
the from-scratch budget; selection happens inside stage two.

The published runs started from BYOL epochs 450, 700, 450, 450 and 300 (folds 0 to 4), read off
their training logs: the checkpoint each BYOL run's worker had picked at the time, by the
review-query score it used before the paper's rank rule. `BYOL_EP` below pins those epochs. They
are not the rank-selected checkpoints the paper's BYOL row reads (250, 550, 300, 500, 250); both
sets are on Hugging Face.

The notebook needs no stage-one file to start. The stage-one cell looks for each fold's checkpoint
in `OUT_DIR`, where a BYOL run of `w9_experiment_5fold_2.ipynb` leaves one every 50 epochs. For a
fold that has none it fetches the file the published run started from, out of
`chironYu/latenarray-i2ce-checkpoints` on Hugging Face, into `OUT_DIR/stage_one`. With
`S1_HUB = ""`, or when a file cannot be fetched, it trains the BYOL arm itself into that folder,
with the flags of `w9_experiment_5fold_2.ipynb`, and stops at the epoch stage two starts from. The
training loop reads `--epochs` only as the place to stop (constant learning rate, seed = fold), so
such a run ends where a 2,000-epoch run of the fold stands at that epoch. Its own folder keeps it
apart from a full BYOL run made later.

`pool` is the store-page prefix plus the game's whole 2,048-sentence review pool, the paper's
2,048-sentence budget. The reference rows, fully coupled I-CE and the windowed teacher from
scratch, are the towers of `w9_final_experiment.ipynb` and `w9_swin_5fold.ipynb`, only read here.

In [ ]:
# constants
import os

REPO = os.path.abspath("..")   # this release folder (contains Pod/ and VICReg_review/)
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir

ARM = "wcle_swin168step84loop2i2ce_icetf"      # w = 168, stride 84, two micro-passes
CAP = "pool"
FOLDS = [0, 1, 2, 3, 4]
EPOCHS = 600
EXTRA_FILES = []

# stage one: the BYOL tower of the same fold, at the epoch the published run started from
BYOL_EP = {0: 450, 1: 700, 2: 450, 3: 450, 4: 300}
S1_ARM = "wcle_byol_bytf"
S1_DIR = os.path.join(OUT_DIR, "stage_one")    # stage-one checkpoints this notebook fetches or trains
S1_HUB = "chironYu/latenarray-i2ce-checkpoints"   # where the published ones are; "" = train them here

def byol_ckpt(k):
    """fold k's stage-one checkpoint: the one a full BYOL run left in OUT_DIR
    (w9_experiment_5fold_2.ipynb), else the one the stage-one cell fetches or trains"""
    f = f"ckpt_w9cv_{S1_ARM}_fold{k}_gpool_fp_ep{BYOL_EP[k]}.pt"
    p = os.path.join(OUT_DIR, f)
    return p if os.path.exists(p) else os.path.join(S1_DIR, f)

# (arm, extra worker args, run-name suffix); the worker adds _bw when it gets --init-ckpt
JOBS = [(ARM, lambda k: ["--init-ckpt", byol_ckpt(k)], "_bw")]
LABELS = ["two-stage: BYOL warm start, then windowed"]
# reference rows, read-only: (label, arm, suffix, cap)
REF_ROWS = [("I-CE, fully coupled teacher", "wcle_i2ce_icetf", "", "pool"),
            ("swin-I-CE from scratch", ARM, "", "pool")]

N_FOLDS = 5                            # the split: fold k = test, fold k+1 = validation
CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 50, 2, 10   # ZS: seeds unused

# VRAM scheduler knobs
SAFETY = 0.85
RESERVE_GIB = 1.5

def xargs(extra, k):
    """a job's extra worker arguments: a list, or a function of the fold"""
    return list(extra(k)) if callable(extra) else list(extra)

def nm_of(r, k, sfx="", cap=None):
    cap = CAP if cap is None else cap
    # as the worker names it: a run on the 512 packs ends in _d512
    return (f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "") + sfx
            + ("_d512" if cap == 512 else ""))

os.makedirs(OUT_DIR, exist_ok=True)
print("jobs:", len(JOBS) * len(FOLDS),
      f"({len(JOBS)} arms x folds {FOLDS}) @{CAP}/{EPOCHS}ep")

In [ ]:
# Local setup (release build: the code ships with this folder -- no
# repository synchronisation is needed or performed).
import importlib.util
import os
import sys
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        %pip -q install scikit-learn scipy
        break
os.chdir(REPO)
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")

In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
REQUIRED += EXTRA_FILES
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# STAGE ONE: the BYOL towers stage two starts from.
# A fold whose checkpoint exists (left in OUT_DIR by a full BYOL run, or in
# S1_DIR by an earlier pass of this cell) is skipped. For the others the
# published checkpoint is fetched from Hugging Face. What cannot be fetched
# is trained here: the BYOL arm with the flags of w9_experiment_5fold_2.ipynb,
# stopped at BYOL_EP[k]. The training loop reads --epochs only as the place
# to stop, so the run ends where a 2,000-epoch run of the fold (seed = fold)
# stands at that epoch. One cost is measured, then the folds share the GPUs.
import os, queue, subprocess, tempfile, threading, time
from pathlib import Path

S1_STAGGER = 60       # seconds between two launches (host-RAM transient at boot)
S1_MAX_CO = 4         # workers per GPU at most

def s1_cmd(k, out_dir, epochs, tail):
    """the five-fold worker's command line for fold k's BYOL run"""
    return (["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
             out_dir, "--repo", REPO, "--arm", S1_ARM, "--fold", str(k),
             "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
             "--epochs", str(epochs), "--full-pool",
             "--full-pool-path", FULL_POOL_PATH] + list(tail))

def s1_fetch(k):
    """fold k's published stage-one checkpoint, from Hugging Face into S1_DIR"""
    from huggingface_hub import hf_hub_download
    hf_hub_download(S1_HUB, os.path.basename(byol_ckpt(k)), local_dir=S1_DIR)

for k in ([k for k in FOLDS if not os.path.exists(byol_ckpt(k))] if S1_HUB else []):
    try:
        s1_fetch(k)
        print(f"[stage one] fold {k}: fetched {os.path.basename(byol_ckpt(k))}")
    except Exception as e:     # no network, no huggingface_hub, no such file
        print(f"[stage one] fold {k}: not fetched ({type(e).__name__}), "
              f"it is trained here instead")

s1_todo = [k for k in FOLDS if not os.path.exists(byol_ckpt(k))]
print("stage one:", f"folds {s1_todo} to train, to epochs "
      f"{[BYOL_EP[k] for k in s1_todo]}" if s1_todo else "every fold has its checkpoint")
s1_fails = []
if s1_todo:
    s1_cdir = Path(S1_DIR) / "claims"
    s1_logd = Path(S1_DIR) / "logs"
    s1_logd.mkdir(parents=True, exist_ok=True)
    s1_meas = os.path.join(tempfile.gettempdir(), "w9_measure_out")
    os.makedirs(s1_meas, exist_ok=True)
    s1_gpus = J.detect_gpus()

    # one measured cost, then as many workers per GPU as its free memory holds
    tf = Path(tempfile.gettempdir()) / "w9cv_vram_stage_one.txt"
    tf.unlink(missing_ok=True)
    with open(s1_logd / "measure_stage_one.log", "w") as fh:
        subprocess.run(s1_cmd(s1_todo[0], s1_meas, 1, ["--measure-vram", str(tf)]),
                       stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=s1_gpus[0]))
    s1_cost = int(tf.read_text()) if tf.exists() else 0
    s1_slots = []
    for g in s1_gpus:
        free = int(subprocess.check_output(
            ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits",
             "-i", str(g)]).decode().strip().split("\n")[0]) * 2**20
        room = int(free * SAFETY - RESERVE_GIB * 2**30)
        s1_slots += [g] * (max(1, min(S1_MAX_CO, room // s1_cost)) if s1_cost else 1)
    print(f"[stage one] cost {s1_cost / 2**30:.2f}G, slots per GPU "
          f"{ {g: s1_slots.count(g) for g in s1_gpus} }", flush=True)

    s1_q = queue.Queue()
    for k in s1_todo:
        s1_q.put(k)

    def s1_run(g, delay):
        time.sleep(delay)
        while True:
            try:
                k = s1_q.get_nowait()
            except queue.Empty:
                return
            nm = nm_of(S1_ARM, k)
            if not J.try_claim(s1_cdir, nm):
                print(f"[claim] {nm} held elsewhere -- skipped", flush=True); continue
            cmd = s1_cmd(k, S1_DIR, BYOL_EP[k],
                         ["--ckpt-every", str(CKPT_EVERY),
                          "--ckpt-seeds", str(CKPT_SEEDS),
                          "--topup-seeds", str(TOPUP_SEEDS),
                          "--claim-file", str(s1_cdir / f"{nm}.claim")])
            t0 = time.time()
            with open(s1_logd / f"{nm}.log", "w") as fh:
                p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                                   env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
            if p.returncode != 0:
                (s1_cdir / f"{nm}.claim").unlink(missing_ok=True); s1_fails.append(nm)
            print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
                  f"to ep{BYOL_EP[k]} [{(time.time() - t0) / 3600:.1f}h]", flush=True)

    s1_ths = [threading.Thread(target=s1_run, args=(g, i * S1_STAGGER))
              for i, g in enumerate(s1_slots[:len(s1_todo)])]
    for t in s1_ths:
        t.start()
    for t in s1_ths:
        t.join()

s1_missing = [byol_ckpt(k) for k in FOLDS if not os.path.exists(byol_ckpt(k))]
if s1_missing:
    raise RuntimeError(
        "stage one is not complete; a run failed here (see S1_DIR/logs) or another "
        "machine holds it (run this cell again when it is done):\n  "
        + "\n  ".join(s1_missing))
print("stage one ready:", *[byol_ckpt(k) for k in FOLDS], sep="\n  ")

In [ ]:
# VRAM-BUDGET SCHEDULER over the (job x fold) grid.
# Warmup measures one cost per job; the ZS-only done marker is
# tower_<nm>_fp_ep{EPOCHS}.npz (a crashed tower auto-continues from its
# newest ckpt / resume bundle).
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir: with the real job name on the
# SHARED volume, a measure could load another machine's resume bundle
# (start_ep >= 1 -> zero steps -> no cost file) or race its zs_traj writes.
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}")

def worker_cmd(r, extra, k, out_dir, epochs, tail):
    """the five-fold worker's command line for one job"""
    return (["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
             out_dir, "--repo", REPO, "--arm", r, "--fold", str(k),
             "--n-folds", str(N_FOLDS), "--anchor-cap", str(CAP),
             "--epochs", str(epochs), "--full-pool",
             "--full-pool-path", FULL_POOL_PATH] + list(tail) + xargs(extra, k))

todo0 = []
for r, extra, sfx in JOBS:
    for k in FOLDS:
        nm = nm_of(r, k, sfx)
        if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
            print(f"[skip] {nm} at {EPOCHS}"); continue
        todo0.append((r, extra, sfx, k, nm))

cost = {}
for r, extra, sfx in JOBS:
    key = r + sfx
    if not any(t[0] == r and t[2] == sfx for t in todo0):
        continue
    # measure ONE cost per job, on its first fold
    tf = Path(tempfile.gettempdir()) / f"w9cv_vram_{key}.txt"
    tf.unlink(missing_ok=True)
    cmd = worker_cmd(r, extra, FOLDS[0], MEAS_OUT, 1, ["--measure-vram", str(tf)])
    print(f"[warmup] {key} ...", flush=True)
    with open(logd / f"measure_cv_{key}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[key] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] {key}: {cost[key] / 2**30:.2f}G", flush=True)

todo = sorted(((r, e, sfx, k, nm, cost[r + sfx])
               for r, e, sfx, k, nm in todo0), key=lambda x: -x[5])
now_used = {g: 0 for g in gpus}
now_cnt = {g: 0 for g in gpus}   # concurrent workers per GPU
MAX_CO = 4        # host-RAM guard: ~8.5G boot transient per worker
LAUNCH_STAGGER = 60
fails = []
cvn = threading.Condition()

def run_job(g, r, extra, sfx, k, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = worker_cmd(r, extra, k, OUT_DIR, EPOCHS,
                         ["--ckpt-every", str(CKPT_EVERY),
                          "--ckpt-seeds", str(CKPT_SEEDS),
                          "--topup-seeds", str(TOPUP_SEEDS),
                          "--claim-file", str(cdir / f"{nm}.claim")])
        t0 = time.time()
        with open(logd / f"{nm}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cvn:
            now_used[g] -= c
            now_cnt[g] -= 1
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, e, sfx, k, nm, c = pending[i]
            fit = [g for g in gpus if now_cnt[g] < MAX_CO
                   and (now_used[g] + c <= budget[g] or now_used[g] == 0)]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            now_cnt[g] += 1
            th = threading.Thread(target=run_job, args=(g, r, e, sfx, k, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G, used "
                  f"{now_used[g] / 2**30:.1f}/{budget[g] / 2**30:.0f}G)", flush=True)
            prog = True
            cvn.wait(timeout=LAUNCH_STAGGER)   # stagger the boots
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"FINAL grid drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)

In [ ]:
# SELECTION-REPAIR: upgrade legacy zsbest from saved projections.
# The selector marker distinguishes paper ranks from old tag scores.
# --eval-only forbids training; ranks use validation neutral/noname.
# For CPU-only reselection from a read-only mirror, use w9_reselect.py.
import json as _json, os, queue as _q, subprocess, threading, time
from pathlib import Path
from w9_selection import SELECTOR

need = []
for r, extra, sfx in JOBS:
    for k in FOLDS:
        nm = nm_of(r, k, sfx)
        if not (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
            continue
        zb = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
        if zb.exists() and _json.loads(zb.read_text()).get("selector") == SELECTOR:
            continue
        need.append((r, extra, k, nm))
print(f"{len(need)} tower(s) need re-selection")
jobs = _q.Queue()
for j in need:
    jobs.put(j)
cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"

def rep(g):
    while True:
        try:
            r, extra, k, nm = jobs.get_nowait()
        except _q.Empty:
            return
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held -- skipped", flush=True); continue
        cmd = worker_cmd(r, extra, k, OUT_DIR, EPOCHS,
                         ["--eval-only", "--claim-file", str(cdir / f"{nm}.claim")])
        t0 = time.time()
        with open(logd / f"reselect_{nm}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        print(f"[gpu{g}] reselect {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 60:.0f}m]", flush=True)

ths = [threading.Thread(target=rep, args=(g,)) for g in J.detect_gpus()]
for t in ths:
    t.start()
for t in ths:
    t.join()
print("re-selection pass done")

In [ ]:
# READOUT: the worker's own reading of each selected checkpoint, against the
# gallery it trained with: mean+-std over the folds, then the paired per-fold
# difference in Stripped hit@1 against the first reference row. The paper's
# tables are the ten-draw readings of eval/k10_eval.py on these checkpoints.
import json
import numpy as np
from pathlib import Path

def _load(nm):
    p = Path(OUT_DIR) / f"zsbest_{nm}_fp.json"
    return json.loads(p.read_text()) if p.exists() else None

ROWS = list(REF_ROWS) + [(lab, r, sfx, CAP) for (r, _e, sfx), lab in zip(JOBS, LABELS)]
KEYS = [("nm_neutral", "name@1"), ("h5_neutral", "name@5"),
        ("nm_noname", "stripped@1"), ("h5_noname", "stripped@5"),
        ("tag_noname", "tag")]
per = {}
for lab, r, sfx, cap in ROWS:
    ds = {k: _load(nm_of(r, k, sfx, cap)) for k in FOLDS}
    per[lab] = {k: d for k, d in ds.items() if d}
    line = f"{lab:36s} [{len(per[lab])}/{len(FOLDS)}]"
    for key, klab in KEYS:
        v = [d[key] for d in per[lab].values() if d.get(key) is not None]
        if v:
            line += f"  {klab} {np.mean(v):.3f}+-{np.std(v):.3f}"
    print(line)
ref = ROWS[0][0]
for lab, *_ in ROWS[1:]:
    both = sorted(set(per[lab]) & set(per[ref]))
    if both:
        ds = [per[lab][k]["nm_noname"] - per[ref][k]["nm_noname"] for k in both]
        print(f"paired stripped@1, {lab} minus {ref}: "
              + " ".join(f"{d:+.3f}" for d in ds)
              + f"  | mean {np.mean(ds):+.3f}  ahead {sum(d > 0 for d in ds)}/{len(ds)}")

In [ ]:
# AUTO-STOP removed in the release build: stopping the machine is cloud-
# provider tooling, not part of the experiment. All results are already on
# the shared volume when the run cells finish.
print("run complete -- results are in", OUT_DIR)